# Hollow magnetic sphere in a uniform applied field — FEniCSx

**Reference:** I. D. Mayergoyz, M. V. K. Chari, J. D'Angelo, *A New Scalar Potential Formulation for Three-Dimensional Magnetostatic Problems*, IEEE Trans. Magn. **23**(6), 3889–3894, 1987.

This notebook is the FEniCSx (dolfinx 0.8 / 0.9 / 0.10) translation of `hollowsphere.m`. It also adds:

1. the paper's actual **two-step "difference potential"** method (Sections III–IV, eqs. 20–49);
2. an **exact solution of the same truncated problem**, so the FE error can be measured rather than eyeballed.

**Requirements:** `dolfinx`, `gmsh` (Python API), `matplotlib`. Run in serial; the plotting cells assume one MPI rank.

## 1. The problem

A spherical shell $a<R<b$ with relative permeability $\mu_r$ sits in a uniform applied field $\mathbf H_s = h_a\,\mathbf e_z$. There are no currents, so $\mathbf H_s$ is the paper's "source" field. Everything is axisymmetric, so we solve on the meridian half-plane $(r,z)$, $r\ge 0$:

| region | definition | $\mu$ | tag |
|---|---|---|---|
| $\Omega_1$ cavity | $R<a$ | $\mu_0$ | 1 |
| $\Omega_2$ shell ("iron", $V^+$) | $a<R<b$ | $\mu_r\mu_0$ | 2 |
| $\Omega_3$ outer air | $b<R<c$ | $\mu_0$ | 3 |

| boundary | where | condition | tag |
|---|---|---|---|
| $S_{in}$ | $R=a$ | interface | 11 |
| $S_{out}$ | $R=b$ | interface | 12 |
| $\tilde S$ (artificial, paper Fig. 2) | $R=c$ | $\phi=0$ | 13 |
| axis | $r=0$ | natural (the $r$-weight kills the flux) | 14 |

Axisymmetry: $dV = 2\pi r\,dr\,dz$ and $dS = 2\pi r\,d\ell$. The $2\pi$ cancels from both sides of every weak form, which leaves the factor $r$. That is the `rc` and `(aa ± bb/3)` in the MATLAB code.

## 2. Imports

In [ ]:
import numpy as np
from mpi4py import MPI
import gmsh
import ufl
import dolfinx
from dolfinx import fem, io
from dolfinx.fem.petsc import LinearProblem
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

## 3. Parameters

Same values as `hollowsphere.m`. Note the renaming: MATLAB `r3` → `a`, `r1` → `b`, `r2` → `c`.

In [ ]:
# ------------------------------------------------------------------ parameters
# (same names / values as hollowsphere.m:  r3 = a, r1 = b, r2 = c)
a = 0.1          # inner radius of shell     (MATLAB r3)
b = 0.2          # outer radius of shell     (MATLAB r1)
c = 0.5          # truncation radius, phi=0  (MATLAB r2)
mu0 = 1.0        # MATLAB muo
mur = 10.0       # MATLAB mur
ha = 100.0       # applied field along z
MU_INF = 1e6     # "infinite" permeability for step 1 of the two-step method

h_interface = 0.002   # MATLAB used Hedge = 0.001 on interfaces; refine as you like
h_far = 0.02

CAV, SHELL, AIR = 1, 2, 3
S_IN, S_OUT, OUTER, AXIS = 11, 12, 13, 14

comm = MPI.COMM_WORLD

## 4. Geometry and mesh (gmsh)

We fragment a rectangle $[0,c]\times[-c,c]$ with three disks of radius $c,b,a$, then keep only the pieces with $r>0$ and $R<c$.

- **Surfaces** are classified by area. The half-disk and two half-annuli have clearly distinct areas, whereas their centroids are ambiguous: the shell's centroid lies at $r\approx0.099<a$.
- **Curves** are classified as axis segments ($r=0$) or semicircular arcs, whose centroid sits at distance $2R/\pi$ from the origin.
- A distance/threshold size field refines the mesh at both interfaces, playing the role of MATLAB's `Hedge`.

The interfaces are tagged as **interior facets**, so the interface integrals run over actual mesh edges. No node-ordering loops are needed, unlike `findNodes` + `Nf(i), Nf(i+1)` in MATLAB.

In [ ]:
# ------------------------------------------------------------------ 1. geometry + mesh (gmsh)
def build_mesh():
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("hollow_sphere_axisym")
    occ = gmsh.model.occ
    if comm.rank == 0:
        rect = occ.addRectangle(0.0, -c, 0.0, c, 2 * c)          # r in [0,c], z in [-c,c]
        disks = [occ.addDisk(0, 0, 0, R, R) for R in (c, b, a)]
        occ.fragment([(2, rect)], [(2, d) for d in disks])
        occ.synchronize()
        # keep only the pieces with r > 0 and inside R < c
        drop = []
        for dim, tag in gmsh.model.getEntities(2):
            x, y, _ = occ.getCenterOfMass(dim, tag)
            if x < 0 or np.hypot(x, y) > c:
                drop.append((dim, tag))
        occ.remove(drop, recursive=True)
        occ.synchronize()

        # classify surfaces by area (half-disk / half-annuli have distinct areas)
        target = {CAV: np.pi * a**2 / 2, SHELL: np.pi * (b**2 - a**2) / 2,
                  AIR: np.pi * (c**2 - b**2) / 2}
        groups = {k: [] for k in target}
        for dim, tag in gmsh.model.getEntities(2):
            A = occ.getMass(dim, tag)
            k = min(target, key=lambda k: abs(target[k] - A))
            groups[k].append(tag)
        for k, tags in groups.items():
            gmsh.model.addPhysicalGroup(2, tags, k)

        # classify curves: axis segments (r=0) or semicircular arcs (COM at |x| = 2R/pi)
        cgroups = {S_IN: [], S_OUT: [], OUTER: [], AXIS: []}
        for dim, tag in gmsh.model.getEntities(1):
            x, y, _ = occ.getCenterOfMass(dim, tag)
            if abs(x) < 1e-8:
                cgroups[AXIS].append(tag)
            else:
                Rarc = np.hypot(x, y) * np.pi / 2
                k = {a: S_IN, b: S_OUT, c: OUTER}[min((a, b, c), key=lambda R: abs(R - Rarc))]
                cgroups[k].append(tag)
        for k, tags in cgroups.items():
            gmsh.model.addPhysicalGroup(1, tags, k)

        # graded size field: fine at both interfaces, coarse far away
        f_d = gmsh.model.mesh.field.add("Distance")
        gmsh.model.mesh.field.setNumbers(f_d, "CurvesList", cgroups[S_IN] + cgroups[S_OUT])
        gmsh.model.mesh.field.setNumber(f_d, "Sampling", 400)
        f_t = gmsh.model.mesh.field.add("Threshold")
        gmsh.model.mesh.field.setNumber(f_t, "InField", f_d)
        gmsh.model.mesh.field.setNumber(f_t, "SizeMin", h_interface)
        gmsh.model.mesh.field.setNumber(f_t, "SizeMax", h_far)
        gmsh.model.mesh.field.setNumber(f_t, "DistMin", 0.005)
        gmsh.model.mesh.field.setNumber(f_t, "DistMax", 0.1)
        gmsh.model.mesh.field.setAsBackgroundMesh(f_t)
        for opt in ("Mesh.MeshSizeExtendFromBoundary", "Mesh.MeshSizeFromPoints",
                    "Mesh.MeshSizeFromCurvature"):
            gmsh.option.setNumber(opt, 0)
        gmsh.model.mesh.generate(2)           # linear triangles, as in the MATLAB code

    # gmsh -> dolfinx (API moved between versions)
    try:                                        # dolfinx >= 0.10
        from dolfinx.io import gmsh as gmshio
        md = gmshio.model_to_mesh(gmsh.model, comm, 0, gdim=2)
        msh, ct, ft = md.mesh, md.cell_tags, md.facet_tags
    except (ImportError, AttributeError):       # dolfinx 0.8 / 0.9
        from dolfinx.io import gmshio
        msh, ct, ft = gmshio.model_to_mesh(gmsh.model, comm, 0, gdim=2)
    gmsh.finalize()
    return msh, ct, ft


msh, ct, ft = build_mesh()
tdim = msh.topology.dim
msh.topology.create_connectivity(tdim - 1, tdim)

## 5. Function spaces, material data, boundary conditions

- $\phi\in P_1$, linear triangles as in the MATLAB code.
- $\mu$ is a piecewise-constant DG0 function built from the cell tags. A second copy uses $\mu^+\to\infty$ for step 1 of the paper's method.
- $\phi=0$ on $\tilde S$ corresponds to deleting the `Ne1` rows and columns in MATLAB.

In [ ]:
# ------------------------------------------------------------------ 2. spaces, material, BCs
V = fem.functionspace(msh, ("Lagrange", 1))            # P1, like the MATLAB code
Q = fem.functionspace(msh, ("DG", 0))                  # piecewise-constant material data


def piecewise(values):
    """DG0 function taking values[tag] on each tagged subdomain."""
    f = fem.Function(Q)
    for tag, val in values.items():
        f.x.array[ct.find(tag)] = val
    return f


mu = piecewise({CAV: mu0, SHELL: mur * mu0, AIR: mu0})          # true permeability
mu_inf = piecewise({CAV: mu0, SHELL: MU_INF * mu0, AIR: mu0})   # step-1 permeability
chi_iron = piecewise({CAV: 0.0, SHELL: 1.0, AIR: 0.0})
chi_air = piecewise({CAV: 1.0, SHELL: 0.0, AIR: 1.0})

# phi = 0 on the artificial outer boundary S~ (MATLAB: Ne1, rows/cols deleted)
outer_dofs = fem.locate_dofs_topological(V, tdim - 1, ft.find(OUTER))
bcs = [fem.dirichletbc(dolfinx.default_scalar_type(0.0), outer_dofs, V)]
# axis r = 0: natural (zero r-weighted flux), nothing to do. Same as the MATLAB code.

x = ufl.SpatialCoordinate(msh)
r, z = x[0], x[1]
Hs = ufl.as_vector((0.0, ha))                  # applied ("source") field, along +z
dx = ufl.Measure("dx", domain=msh, subdomain_data=ct)
dS = ufl.Measure("dS", domain=msh, subdomain_data=ft)   # interior facets (interfaces)

phi_t, psi = ufl.TrialFunction(V), ufl.TestFunction(V)


def lin_solve(a_form, L_form, prefix):
    opts = {"ksp_type": "preonly", "pc_type": "lu"}
    try:                                       # dolfinx >= 0.10 needs a prefix
        prob = LinearProblem(a_form, L_form, bcs=bcs, petsc_options=opts,
                             petsc_options_prefix=prefix)
    except TypeError:
        prob = LinearProblem(a_form, L_form, bcs=bcs, petsc_options=opts)
    out = prob.solve()
    return out[0] if isinstance(out, tuple) else out


def stiffness(mu_fun):
    # int mu grad(psi).grad(phi) r dA   == eq. (43)/(47) LHS; MATLAB: mu*rc*(Bp'*Bp)*Area
    return mu_fun * ufl.inner(ufl.grad(phi_t), ufl.grad(psi)) * r * dx


def interface_rhs(dmu):
    """(mu+ - mu0) oint_S psi Hs.n_iron r dl  -- eq. (43), the MATLAB fglobal loops.
    n_iron = +e_R on S_out, -e_R on S_in; it is taken analytically because the sign of
    dolfinx's interior-facet normal n('+') is not tied to a material side."""
    xp = x("+")
    Rp = ufl.sqrt(xp[0] ** 2 + xp[1] ** 2)
    eR = xp / Rp
    flux = ufl.dot(Hs, eR) * psi("+") * xp[0]
    return dmu * flux * dS(S_OUT) - dmu * flux * dS(S_IN)

## 6a. Reduced scalar potential — what `hollowsphere.m` solves

Paper eqs. (9)–(14):

$$\mathbf H = \mathbf H_s - \nabla\phi, \qquad \nabla\cdot(\mu\nabla\phi)=\nabla\cdot(\mu\mathbf H_s)=0 \ \text{in each region},$$
$$[\phi]=0,\qquad \mu^+\frac{\partial\phi^+}{\partial n}-\mu_0\frac{\partial\phi^-}{\partial n}=(\mu^+-\mu_0)\,\mathbf n\cdot\mathbf H_s \quad\text{on } S,$$

where $\mathbf n$ points **out of the iron**. Multiplying by a test function $\psi$, integrating by parts in each region and using the jump condition gives eq. (43):

$$\int_V \mu\,\nabla\psi\cdot\nabla\phi\; r\,dA = (\mu^+-\mu_0)\oint_S \psi\,\mathbf H_s\cdot\mathbf n\; r\,d\ell .$$

The shell has **two** interfaces:

| interface | $\mathbf n$ out of iron | RHS density | MATLAB |
|---|---|---|---|
| $S_{out}$, $R=b$ | $+\mathbf e_R$ | $+(\mu_r-1)\,h_a\cos\theta$ | `Nf1`, factor `(mur-1)` |
| $S_{in}$, $R=a$ | $-\mathbf e_R$ | $-(\mu_r-1)\,h_a\cos\theta$ | `Nf2`, factor `(1-mur)` |

with $\cos\theta = z/R$. dolfinx's interior-facet normal `n('+')` has no fixed material side, so $\mathbf e_R=\mathbf x/R$ is used analytically.

**Volume-equivalent cross-check.** Since $\nabla\cdot\mathbf H_s=0$, the divergence theorem gives
$$(\mu^+-\mu_0)\oint_S\psi\,\mathbf H_s\cdot\mathbf n\,dS=\int_{V^+}(\mu^+-\mu_0)\,\mathbf H_s\cdot\nabla\psi\,dV.$$
That form needs neither normals nor facets. Both versions are solved, and they must agree.

In [ ]:
# ------------------------------------------------------------------ 3a. reduced potential (= MATLAB)
phi = lin_solve(stiffness(mu), interface_rhs((mur - 1.0) * mu0), "red_")
phi.name = "phi_reduced"

# same problem, volume form of the RHS (cross-check of normals / signs / r-weights)
phi_vol = lin_solve(stiffness(mu), (mu - mu0) * ufl.dot(Hs, ufl.grad(psi)) * r * dx, "vol_")

H_red = Hs - ufl.grad(phi)                     # eq. (9);  MATLAB: hx = -dphi/dx, hy = ha - dphi/dy

## 6b. The paper's two-step "difference potential" method

**Why the paper exists.** Inside the iron, eq. (9) computes $\mathbf H^+$ as $\mathbf H_s-\nabla\phi$. When $\mu^+\gg\mu_0$ these two terms are nearly equal and opposite, so the result is a small difference of large numbers. This cancellation error is fatal for nonlinear, saturating iron.

**Step 1.** Solve (43) with $\mu^+\to\infty$. This gives $\mathbf H_0=\mathbf H_s-\nabla\phi_0$ in air and $\mathbf H_0^+=0$ in iron (eq. 20).

**Step 2.** The difference field $\mathbf h=\mathbf H-\mathbf H_0=-\nabla\tilde\phi$ satisfies eqs. (33)–(36), with weak form (47):
$$\int_V\mu\,\nabla\psi\cdot\nabla\tilde\phi\,dV=-\mu_0\oint_S\psi\,\mathbf H_0\cdot\mathbf n\,dS .$$
Apply the divergence theorem in the air region $V^-$, using $\nabla\cdot\mathbf H_0=0$, $\psi=0$ on $\tilde S$, and $\mathbf n_{air}=-\mathbf n_{iron}$ on $S$:
$$-\mu_0\oint_S\psi\,\mathbf H_0\cdot\mathbf n_{iron}\,dS=+\mu_0\int_{V^-}\mathbf H_0\cdot\nabla\psi\,dV .$$

**Total field:** $\mathbf H=\mathbf H_0+\mathbf h$. In the iron $\mathbf H^+=-\nabla\tilde\phi$ (eqs. 25, 37), with no subtraction.

Both steps reuse the same stiffness operator; only $\mu^+$ and the RHS change. This is the paper's "assemble once" remark. For a *linear* shell ($\mu_r=10$) both routes agree to discretisation accuracy. The two-step method pays off only for very large or field-dependent $\mu^+$.

In [ ]:
# ------------------------------------------------------------------ 3b. paper's two-step method
# step 1: mu+ -> infinity
phi0 = lin_solve(stiffness(mu_inf), interface_rhs((MU_INF - 1.0) * mu0), "s1_")
H0 = chi_air * (Hs - ufl.grad(phi0))          # H0 in air, H0+ = 0 in iron (eq. 20)
# step 2: difference potential, eq. (47) in its volume-equivalent form
phi_d = lin_solve(stiffness(mu), mu0 * chi_air * ufl.dot(H0, ufl.grad(psi)) * r * dx, "s2_")
phi_d.name = "phi_difference"
H_two = H0 - ufl.grad(phi_d)                  # H = H0 + h;  in iron: H = -grad(phi~), no cancellation

## 7. Exact solution of the truncated problem and checks

Write the total potential $\Phi=-h_a z+\phi$, so $\mathbf H=-\nabla\Phi$, and take $\Phi=f(R)\cos\theta$ with
$$f_1=A_1R,\qquad f_2=A_2R+\frac{B_2}{R^2},\qquad f_3=A_3R+\frac{B_3}{R^2}.$$
Impose continuity of $\Phi$ and of $\mu\,\partial\Phi/\partial R$ at $R=a,b$, and $\Phi(c)=-h_a c$ (i.e. $\phi=0$ on $\tilde S$). That is five linear equations for five constants. The cavity field is uniform: $H_z=-A_1$.

For comparison, the infinite-domain ($c\to\infty$) shielding result is
$$\frac{H_z^{cav}}{h_a}=\frac{9\mu_r}{(2\mu_r+1)(\mu_r+2)-2(a/b)^3(\mu_r-1)^2}.$$
The gap between the two exact values is the error caused by the artificial boundary $\tilde S$ (paper Section V). For these parameters it is about 5%, which is larger than the discretisation error.

**Expected output** (independently checked with the same weak forms): relative $\phi$ error $\sim10^{-4}$–$10^{-3}$ and $O(h^2)$; surface-vs-volume and two-step-vs-reduced differences $\sim10^{-5}$; cavity $H_z\approx 40.69$ against $38.83$ for the infinite domain.

In [ ]:
# ------------------------------------------------------------------ 4. exact solution + checks
def exact_coeffs(murel):
    M = np.array([[a, -a, -1 / a**2, 0, 0],
                  [mu0, -murel, 2 * murel / a**3, 0, 0],
                  [0, b, 1 / b**2, -b, -1 / b**2],
                  [0, murel, -2 * murel / b**3, -1, 2 / b**3],
                  [0, 0, 0, c, 1 / c**2]])
    return np.linalg.solve(M, [0, 0, 0, 0, -ha * c])


A1, A2, B2, A3, B3 = exact_coeffs(mur)


def phi_exact(rr, zz):
    R = np.maximum(np.hypot(rr, zz), 1e-30)
    f = np.where(R < a, A1 * R, np.where(R < b, A2 * R + B2 / R**2, A3 * R + B3 / R**2))
    return f * zz / R + ha * zz                # reduced potential: phi = Phi_tot + ha z


xyz = V.tabulate_dof_coordinates()
pe = phi_exact(xyz[:, 0], xyz[:, 1])
scale = np.abs(pe).max()


def assemble(expr):
    return comm.allreduce(fem.assemble_scalar(fem.form(expr)), op=MPI.SUM)


def cavity_Hz(H):                              # r-weighted (= 3D volume) average over the cavity
    return assemble(H[1] * r * dx(CAV)) / assemble(r * dx(CAV))


def diff_L2(H1, H2):
    return np.sqrt(assemble(ufl.inner(H1 - H2, H1 - H2) * r * dx) / assemble(ha**2 * r * dx))


Hz_inf = ha * 9 * mur / ((2 * mur + 1) * (mur + 2) - 2 * (a / b) ** 3 * (mur - 1) ** 2)
if comm.rank == 0:
    print(f"dofs = {V.dofmap.index_map.size_global}")
print(f"max|phi_FE - phi_exact| / max|phi_exact|  : "
      f"{np.abs(phi.x.array - pe).max() / scale:.3e}")
print(f"surface-RHS vs volume-RHS  (max rel diff)  : "
      f"{np.abs(phi.x.array - phi_vol.x.array).max() / scale:.3e}")
print(f"two-step vs reduced, ||H_two-H_red||/ha    : {diff_L2(H_two, H_red):.3e}")
print(f"cavity Hz  reduced FE  = {cavity_Hz(H_red):8.3f}")
print(f"cavity Hz  two-step FE = {cavity_Hz(H_two):8.3f}")
print(f"cavity Hz  exact (truncated at R=c)   = {-A1:8.3f}")
print(f"cavity Hz  exact (infinite domain)    = {Hz_inf:8.3f}   <- truncation error of S~")

## 8. Post-processing

- **Element-wise field (DG0).** For P1 elements $\nabla\phi$ is constant per triangle, so this is exact. It corresponds to MATLAB's `hx(i), hy(i)`.
- **Nodal field (P1).** This is an $r$-weighted $L^2$ projection, the analogue of MATLAB's patch averaging.

Caveat: $H_r$ jumps by a factor $\mu_r$ across each interface, because $B_n$ is continuous. Any continuous nodal field smears that jump, so use the element-wise field near $S$. The fields are also written to XDMF for ParaView.

In [ ]:
# ------------------------------------------------------------------ 5. post-processing
# (i) element-wise field (exact for P1: grad is constant per triangle). This is the
#     MATLAB `hx(i), hy(i)` before nodal averaging.
W0 = fem.functionspace(msh, ("DG", 0, (2,)))
ip = W0.element.interpolation_points
ip = ip() if callable(ip) else ip
H_dg = fem.Function(W0, name="H_elementwise")
H_dg.interpolate(fem.Expression(H_red, ip))

# (ii) nodal field (the MATLAB patch-averaging step). An L2 projection onto P1.
#     Caveat: H_r = H.e_r jumps by a factor mu_r across S, so any continuous nodal field
#     smears that jump. Use the element-wise field when looking at the interfaces.
W1 = fem.functionspace(msh, ("Lagrange", 1, (2,)))
u_, v_ = ufl.TrialFunction(W1), ufl.TestFunction(W1)
try:
    proj = LinearProblem(ufl.inner(u_, v_) * r * dx, ufl.inner(H_red, v_) * r * dx,
                         petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
                         petsc_options_prefix="proj_")
except TypeError:
    proj = LinearProblem(ufl.inner(u_, v_) * r * dx, ufl.inner(H_red, v_) * r * dx,
                         petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
H_nodal = proj.solve()
H_nodal = H_nodal[0] if isinstance(H_nodal, tuple) else H_nodal
H_nodal.name = "H_nodal"

# ParaView output
with io.XDMFFile(comm, "hollow_sphere.xdmf", "w") as xf:
    xf.write_mesh(msh)
    xf.write_function(phi)
    xf.write_function(phi_d)
    xf.write_function(H_dg)
    xf.write_function(H_nodal)

## 9. Field maps

The half-plane is mirrored $r\to-r$ to show the full meridian section. $H_r$ is odd under the mirror.

In [ ]:
if comm.size == 1:
    pts = msh.geometry.x[:, :2]
    cells = np.asarray(msh.geometry.dofmap).reshape(-1, 3)
    cdofs = np.asarray(W0.dofmap.list).reshape(-1)
    Hc = H_dg.x.array.reshape(-1, 2)[cdofs]     # per-cell (H_r, H_z)
    tri = mtri.Triangulation(np.r_[pts[:, 0], -pts[:, 0]], np.r_[pts[:, 1], pts[:, 1]],
                             np.r_[cells, cells + len(pts)])
    th = np.linspace(0, 2 * np.pi, 400)
    fig, axs = plt.subplots(1, 3, figsize=(16, 5.2))
    for ax, vals, title in [(axs[0], np.r_[Hc[:, 1], Hc[:, 1]], r"$H_z$ (element-wise)"),
                            (axs[1], np.r_[Hc[:, 0], -Hc[:, 0]], r"$H_r$ (element-wise)"),
                            (axs[2], np.r_[np.hypot(*Hc.T), np.hypot(*Hc.T)], r"$|H|$")]:
        tp = ax.tripcolor(tri, facecolors=vals, cmap="jet", shading="flat")
        for R in (a, b):
            ax.plot(R * np.cos(th), R * np.sin(th), "k-", lw=0.6)
        ax.set_aspect("equal"); ax.set_title(title); ax.set_xlabel("r"); ax.set_ylabel("z")
        fig.colorbar(tp, ax=ax, shrink=0.8)
    fig.tight_layout(); fig.savefig("hollow_sphere_H.png", dpi=150); plt.show()

## 10. $H_z$ along the symmetry axis: FE vs exact

In [ ]:
# H_z along the symmetry axis, FE vs exact
zz = np.linspace(-c, c, 801)
Rz = np.abs(zz)
Hz_ex = np.where(Rz < a, -A1, np.where(Rz < b, -(A2 - 2 * B2 / np.maximum(Rz, 1e-30) ** 3),
                                       -(A3 - 2 * B3 / np.maximum(Rz, 1e-30) ** 3)))
from dolfinx import geometry
bb = geometry.bb_tree(msh, tdim)
P = np.c_[np.full_like(zz, 1e-6), zz, np.zeros_like(zz)]
coll = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(bb, P), P)
cell_of = np.array([coll.links(i)[0] for i in range(len(zz))])
Hz_fe = H_dg.x.array.reshape(-1, 2)[cdofs[cell_of], 1]
plt.figure(figsize=(7, 4))
plt.plot(zz, Hz_ex, "k-", lw=2, label="exact (truncated domain)")
plt.plot(zz, Hz_fe, "r--", label="FEniCSx, reduced potential")
for s in (-b, -a, a, b):
    plt.axvline(s, color="0.7", lw=0.8)
plt.xlabel("z  (on axis r = 0)"); plt.ylabel(r"$H_z$"); plt.legend(); plt.tight_layout()
plt.savefig("hollow_sphere_axis.png", dpi=150); plt.show()
print("wrote hollow_sphere_H.png, hollow_sphere_axis.png, hollow_sphere.xdmf")